# Document corpus exploratory data analysis

This notebook examines the municipal policy document corpus to understand its structure,
vocabulary, and characteristics before building retrieval models.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

from src.data_loader import load_documents, build_chunk_index, preprocess_text

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

GOLD = '#E8C230'
NAVY = '#3B6FD4'

documents = load_documents('../data/documents.json')
print(f'Loaded {len(documents)} documents')
print(f'Document IDs: {[d["doc_id"] for d in documents]}')

## 1. Document overview

Each document covers a different area of municipal policy. We start by examining basic statistics: character count, word count, and sentence count per document.

In [ ]:
doc_stats = []
for doc in documents:
    text = doc['text']
    doc_stats.append({
        'doc_id': doc['doc_id'],
        'title': doc['title'],
        'characters': len(text),
        'words': len(text.split()),
        'sentences': text.count('.') + text.count('!') + text.count('?'),
        'avg_word_length': np.mean([len(w) for w in text.split()]),
    })

df_stats = pd.DataFrame(doc_stats)
print(df_stats.to_string(index=False))
print(f'\nTotal words across corpus: {df_stats["words"].sum():,}')
print(f'Average document length: {df_stats["words"].mean():.0f} words')

## 2. Document length distributions

Understanding the variation in document lengths helps us choose an appropriate chunk size. If documents are uniformly long, a larger chunk size may work. If they vary widely, smaller chunks with overlap prevent information loss at boundaries.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].barh(df_stats['doc_id'], df_stats['characters'], color=NAVY, alpha=0.8)
axes[0].set_xlabel('Characters')
axes[0].set_title('Document length (characters)')

axes[1].barh(df_stats['doc_id'], df_stats['words'], color=GOLD, alpha=0.8)
axes[1].set_xlabel('Words')
axes[1].set_title('Document length (words)')

axes[2].barh(df_stats['doc_id'], df_stats['sentences'], color=NAVY, alpha=0.6)
axes[2].set_xlabel('Sentences')
axes[2].set_title('Document length (sentences)')

plt.tight_layout()
plt.savefig('../figures/doc_length_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Vocabulary analysis

We examine the most frequent terms across the corpus after basic preprocessing. High-frequency terms that appear in many documents are less useful for retrieval because they do not discriminate between documents. Terms that are specific to one or two documents are the most informative for matching queries to the right passage.

In [ ]:
# Build corpus-wide vocabulary
all_words = []
for doc in documents:
    cleaned = preprocess_text(doc['text'])
    all_words.extend(cleaned.split())

word_counts = Counter(all_words)
print(f'Total tokens: {len(all_words):,}')
print(f'Unique tokens: {len(word_counts):,}')
print(f'\nTop 30 most frequent terms:')
for word, count in word_counts.most_common(30):
    print(f'  {word:20s} {count:4d}')

In [ ]:
# Plot top 30 terms excluding stopwords
stopwords = {'the', 'and', 'of', 'to', 'in', 'a', 'for', 'with', 'on', 'is', 'by', 'that',
             'from', 'are', 'at', 'an', 'or', 'as', 'this', 'be', 'it', 'has', 'its', 'their',
             'through', 'each', 'per', 'all', 'also', 'than', 'no', 'not', 'more'}
filtered_counts = {w: c for w, c in word_counts.items() if w not in stopwords and len(w) > 2}
top_terms = Counter(filtered_counts).most_common(30)

fig, ax = plt.subplots(figsize=(10, 8))
terms, counts = zip(*top_terms)
ax.barh(range(len(terms)), counts, color=GOLD, alpha=0.85)
ax.set_yticks(range(len(terms)))
ax.set_yticklabels(terms)
ax.invert_yaxis()
ax.set_xlabel('Frequency')
ax.set_title('Top 30 terms (excluding stopwords)')
plt.tight_layout()
plt.savefig('../figures/top_terms.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Term frequency per document

Some terms appear in many documents ("city", "calgary") and carry low discriminative power. Others are specific to a single policy area. This document-term frequency analysis shows which terms are shared across the corpus and which are unique identifiers.

In [ ]:
# Document frequency: how many documents contain each term
doc_freq = Counter()
for doc in documents:
    unique_terms = set(preprocess_text(doc['text']).split())
    for term in unique_terms:
        if term not in stopwords and len(term) > 2:
            doc_freq[term] += 1

# Terms appearing in only 1 document vs many
df_dist = Counter(doc_freq.values())
print('Document frequency distribution:')
for freq, count in sorted(df_dist.items()):
    print(f'  Appears in {freq:2d} document(s): {count:3d} unique terms')

print(f'\nTerms in all 15 docs: {sum(1 for t, f in doc_freq.items() if f == 15)}')
print(f'Terms in only 1 doc:  {sum(1 for t, f in doc_freq.items() if f == 1)}')

In [ ]:
# Heatmap: top terms vs documents
key_terms = [t for t, _ in Counter(filtered_counts).most_common(20)]
term_doc_matrix = np.zeros((len(key_terms), len(documents)))

for j, doc in enumerate(documents):
    words = preprocess_text(doc['text']).split()
    word_count = Counter(words)
    for i, term in enumerate(key_terms):
        term_doc_matrix[i, j] = word_count.get(term, 0)

fig, ax = plt.subplots(figsize=(14, 8))
im = ax.imshow(term_doc_matrix, cmap='YlOrBr', aspect='auto')
ax.set_xticks(range(len(documents)))
ax.set_xticklabels([d['doc_id'] for d in documents], rotation=45, ha='right')
ax.set_yticks(range(len(key_terms)))
ax.set_yticklabels(key_terms)
ax.set_title('Term frequency heatmap (top 20 terms x 15 documents)')
plt.colorbar(im, ax=ax, label='Term count')
plt.tight_layout()
plt.savefig('../figures/term_doc_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Chunk length distributions

Chunking splits each document into overlapping segments. The chunk size and overlap parameters directly affect retrieval quality. We visualize how different chunk sizes produce different numbers of chunks and different length distributions.

In [ ]:
# Default chunking: 500 chars, 50 overlap
chunks_500, meta_500 = build_chunk_index(documents, chunk_size=500, overlap=50)
print(f'Chunk size=500, overlap=50: {len(chunks_500)} chunks')

# Compare different chunk sizes
for cs in [200, 300, 500, 800, 1000]:
    c, m = build_chunk_index(documents, chunk_size=cs, overlap=50)
    lengths = [len(ch) for ch in c]
    print(f'  chunk_size={cs:4d}: {len(c):3d} chunks, '
          f'avg={np.mean(lengths):.0f}, min={min(lengths)}, max={max(lengths)}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, cs, color in zip(axes, [200, 500, 1000], [NAVY, GOLD, NAVY]):
    c, _ = build_chunk_index(documents, chunk_size=cs, overlap=50)
    lengths = [len(ch) for ch in c]
    ax.hist(lengths, bins=20, color=color, alpha=0.7, edgecolor='white')
    ax.set_title(f'Chunk size = {cs}')
    ax.set_xlabel('Characters per chunk')
    ax.set_ylabel('Count')
    ax.axvline(np.mean(lengths), color='red', linestyle='--', alpha=0.7, label=f'Mean: {np.mean(lengths):.0f}')
    ax.legend()

plt.suptitle('Chunk length distributions at different chunk sizes', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('../figures/chunk_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Chunks per document

The number of chunks per document depends on the document length and the chunk size. Longer documents produce more chunks, which gives them more "chances" to be retrieved. This is worth tracking because it can bias retrieval toward longer documents.

In [ ]:
chunks_per_doc = {}
for m in meta_500:
    doc_id = m['doc_id']
    chunks_per_doc[doc_id] = chunks_per_doc.get(doc_id, 0) + 1

fig, ax = plt.subplots(figsize=(10, 6))
doc_ids = list(chunks_per_doc.keys())
counts = list(chunks_per_doc.values())
colors = [GOLD if c == max(counts) else NAVY for c in counts]
ax.bar(doc_ids, counts, color=colors, alpha=0.8)
ax.set_xlabel('Document ID')
ax.set_ylabel('Number of chunks')
ax.set_title('Chunks per document (chunk_size=500, overlap=50)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('../figures/chunks_per_doc.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Word length and sentence structure

The average word length and sentence count per document give us a sense of language complexity. Municipal policy documents tend to use longer words (technical and legal terminology) compared to conversational text.

In [ ]:
all_word_lengths = [len(w) for doc in documents for w in doc['text'].split()]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].hist(all_word_lengths, bins=range(1, 20), color=NAVY, alpha=0.7, edgecolor='white')
axes[0].set_xlabel('Word length (characters)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Word length distribution across corpus')
axes[0].axvline(np.mean(all_word_lengths), color=GOLD, linestyle='--',
                label=f'Mean: {np.mean(all_word_lengths):.1f}')
axes[0].legend()

axes[1].bar(df_stats['doc_id'], df_stats['avg_word_length'], color=GOLD, alpha=0.8)
axes[1].set_xlabel('Document ID')
axes[1].set_ylabel('Average word length')
axes[1].set_title('Average word length per document')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 8. Evaluation Q&A overview

We have 30 evaluation questions, two per document. Each question has a ground truth set of relevant document IDs. Most questions map to a single document, while a few map to multiple documents that share related content.

In [ ]:
from src.data_loader import load_eval_qa

eval_qa = load_eval_qa('../data/eval_qa.json')
print(f'Evaluation Q&A pairs: {len(eval_qa)}')
print(f'\nSample questions:')
for qa in eval_qa[:5]:
    print(f'  Q: {qa["question"][:70]}...')
    print(f'  Relevant: {qa["relevant_doc_ids"]}')
    print()

# Questions per document
qa_per_doc = Counter()
for qa in eval_qa:
    for doc_id in qa['relevant_doc_ids']:
        qa_per_doc[doc_id] += 1

print('Questions per document:')
for doc_id in sorted(qa_per_doc.keys()):
    print(f'  {doc_id}: {qa_per_doc[doc_id]} questions')

## 9. Question vocabulary overlap with documents

For retrieval to work, the query terms must overlap with the terms in the relevant document. We measure term overlap between each question and its ground truth document to understand how much lexical matching we can expect.

In [ ]:
doc_map = {d['doc_id']: d for d in documents}
overlap_ratios = []

for qa in eval_qa:
    q_terms = set(preprocess_text(qa['question']).split()) - stopwords
    for doc_id in qa['relevant_doc_ids']:
        d_terms = set(preprocess_text(doc_map[doc_id]['text']).split()) - stopwords
        if q_terms:
            ratio = len(q_terms & d_terms) / len(q_terms)
            overlap_ratios.append(ratio)

fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(overlap_ratios, bins=15, color=GOLD, alpha=0.8, edgecolor='white')
ax.axvline(np.mean(overlap_ratios), color='red', linestyle='--',
           label=f'Mean: {np.mean(overlap_ratios):.2f}')
ax.set_xlabel('Term overlap ratio (question vs relevant document)')
ax.set_ylabel('Count')
ax.set_title('Query-document term overlap distribution')
ax.legend()
plt.tight_layout()
plt.show()

print(f'Mean overlap: {np.mean(overlap_ratios):.2f}')
print(f'Min overlap:  {np.min(overlap_ratios):.2f}')
print(f'Max overlap:  {np.max(overlap_ratios):.2f}')

## 10. Summary

Key findings from the EDA:

- **Corpus size**: 15 documents, roughly 150-180 words each, covering distinct municipal policy areas
- **Vocabulary**: The corpus contains a mix of shared administrative terms ("city", "calgary", "program") and domain-specific terms that help distinguish documents
- **Chunking**: With chunk_size=500 and overlap=50, each document produces 2-3 chunks, giving us approximately 35-40 total chunks to search
- **Query-document overlap**: Most evaluation questions share 50-80% of their content terms with the relevant document, suggesting that TF-IDF and BM25 should work well for this corpus
- **Potential challenges**: Questions about topics that span multiple policy areas (e.g., transportation appears in both the transit plan and the infrastructure plan) may be harder to retrieve correctly

Next steps: Feature engineering notebook will build TF-IDF vectors and BM25 indices for the chunked corpus.